In [0]:
""" Utiities for Ingestion, transformation and enrichment """

import logging
from datetime import datetime
import json

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window as W
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType


In [0]:
def get_logger(name="ecommerce_pipeline", level=logging.INFO):
    """Return a configured logger."""
    logger = logging.getLogger(name)
    if not logger.hasHandlers():
        logger.setLevel(level)
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S",
        ))
        logger.addHandler(handler)
    return logger

In [0]:
logger = get_logger()

In [0]:
def parse_config_from_json(json_path):
    """Read a JSON file and return its contents as a dict."""
    
    try:
        with open(json_path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(f"JSON file not found: {json_path}")
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Failed to parse JSON '{json_path}': {e}")
    except Exception as e:
        raise RuntimeError(f"Failed to read JSON '{json_path}': {e}")

In [0]:
# ---- Delta Table Writer ---- #

def delta_writer(
    df: DataFrame,
    table_name: str,
    mode: str = "overwrite",
    partition_by: list = None,
) -> None:
    """Write a DataFrame to a Delta table."""
    
    if df is None:
        raise ValueError("Cannot write a None DataFrame.")
    if not table_name or not table_name.strip():
        raise ValueError("table_name must be a non-empty string.")

    try:
        logger.info("Writing to %s (mode=%s)", table_name, mode)
        writer = (
            df.write
            .mode(mode)
            .option("overwriteSchema", "true")
            .format("delta")
        )
        if partition_by:
            writer = writer.partitionBy(*partition_by)
        writer.saveAsTable(table_name)
        logger.info("Write complete: %s", table_name)
    except Exception as e:
        raise RuntimeError(f"Failed to write Delta table '{table_name}': {e}") from e